# Final chronological run-level split

This notebook materializes the approved provisional run assignment into train, validation, and test CSV files. It preserves all existing features and performs no preprocessing or modeling.

### 1. Define paths and fingerprint both inputs

**What the cell does:** Imports libraries, defines repository-relative paths, and records SHA-256 checksums for the feature dataset and run-candidate report.  
**Why it is important:** Final splitting must be reproducible and must not alter either source file.  
**What to understand:** The displayed fingerprints identify the exact files used and will be checked again after all outputs are written.

In [1]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
FEATURE_INPUT_PATH = PROJECT_ROOT / "data" / "interim" / "feature_dataset.csv"
RUN_CANDIDATES_PATH = PROJECT_ROOT / "reports" / "run_split_candidates.csv"
TRAIN_OUTPUT_PATH = PROJECT_ROOT / "data" / "modeling" / "train.csv"
VALIDATION_OUTPUT_PATH = PROJECT_ROOT / "data" / "modeling" / "validation.csv"
TEST_OUTPUT_PATH = PROJECT_ROOT / "data" / "modeling" / "test.csv"
SUMMARY_OUTPUT_PATH = PROJECT_ROOT / "reports" / "final_split_summary.csv"
MACHINE_OUTPUT_PATH = PROJECT_ROOT / "reports" / "final_split_by_machine.csv"
RUN_OUTPUT_PATH = PROJECT_ROOT / "reports" / "final_split_by_run.csv"

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as file_handle:
        for chunk in iter(lambda: file_handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

input_paths = [FEATURE_INPUT_PATH, RUN_CANDIDATES_PATH]
missing_inputs = [str(path) for path in input_paths if not path.is_file()]
if missing_inputs:
    raise FileNotFoundError(f"Required inputs are missing: {missing_inputs}")
input_hashes_before = {path: sha256_file(path) for path in input_paths}
for path, fingerprint in input_hashes_before.items():
    print(f"{path}: {fingerprint}")

/Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/data/interim/feature_dataset.csv: 29f6dba2b376bd26a586b6ad332a2db5bf043e5cad64f92f658b34fc85c9dc67
/Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/reports/run_split_candidates.csv: f0fa20297eb72f9072f2e2711a53245721c9c0c67faaa2cbd4d8352f299b9858


### 2. Load and validate the feature rows and run metadata

**What the cell does:** Loads both inputs, verifies required columns, safely parses timestamps, and checks that row identity is unique.  
**Why it is important:** Every feature row must be assigned exactly once and chronological checks require valid timestamps.  
**What to understand:** The feature table contains all ten complete candidate runs with no ambiguous machine/run/segment/timestamp row keys.

In [2]:
feature_input = pd.read_csv(FEATURE_INPUT_PATH)
run_candidates = pd.read_csv(RUN_CANDIDATES_PATH)
required_feature_columns = {"machine_id", "run_id", "segment_id", "timestamp", "slowdown_in_5min"}
required_run_columns = {"machine_id", "run_id", "start_timestamp", "end_timestamp", "chronological_order_within_machine"}
missing_feature_columns = required_feature_columns - set(feature_input.columns)
missing_run_columns = required_run_columns - set(run_candidates.columns)
if missing_feature_columns or missing_run_columns:
    raise KeyError({
        "missing_feature_columns": sorted(missing_feature_columns),
        "missing_run_columns": sorted(missing_run_columns),
    })

original_columns = feature_input.columns.tolist()
split_work = feature_input.copy(deep=True)
split_work["_source_row"] = np.arange(len(split_work))
split_work["_timestamp_dt"] = pd.to_datetime(split_work["timestamp"], errors="coerce", utc=True)
run_candidates["_start_dt"] = pd.to_datetime(run_candidates["start_timestamp"], errors="coerce", utc=True)
run_candidates["_end_dt"] = pd.to_datetime(run_candidates["end_timestamp"], errors="coerce", utc=True)

if split_work["_timestamp_dt"].isna().any() or run_candidates[["_start_dt", "_end_dt"]].isna().any().any():
    raise ValueError("Invalid timestamps prevent chronological splitting.")
row_identity_columns = ["machine_id", "run_id", "segment_id", "timestamp"]
assert not split_work.duplicated(row_identity_columns).any(), "Feature rows have duplicate sequence/timestamp identities."
assert run_candidates["run_id"].is_unique and len(run_candidates) == 10
assert set(split_work["run_id"].unique()) == set(run_candidates["run_id"])

print(f"Feature rows: {len(split_work):,}")
print(f"Candidate runs: {len(run_candidates)}")
print(f"Machines: {split_work['machine_id'].nunique()}")

Feature rows: 26,036
Candidate runs: 10
Machines: 3


### 3. Resolve shortened prefixes to exact run IDs

**What the cell does:** Matches each requested prefix against the candidate report and requires exactly one complete run-ID match before building the assignment map.  
**Why it is important:** Relying on abbreviated identifiers could silently assign the wrong run if prefixes collide.  
**What to understand:** The displayed lists are the exact full IDs that will be used for train, validation, and test.

In [3]:
requested_prefixes = {
    "train": ["89cdc34b", "d88b15dd", "ec61755d", "2e9f4457", "f24e9c1a"],
    "validation": ["fac82c2e", "373070d1"],
    "test": ["6835f125", "19127a70", "70577f8e"],
}

def resolve_prefix(prefix):
    matches = run_candidates.loc[run_candidates["run_id"].astype(str).str.startswith(prefix), "run_id"].tolist()
    if len(matches) != 1:
        raise ValueError(f"Prefix {prefix!r} matched {len(matches)} run IDs: {matches}")
    return matches[0]

exact_run_ids = {
    split_name: [resolve_prefix(prefix) for prefix in prefixes]
    for split_name, prefixes in requested_prefixes.items()
}
run_to_split = {
    run_id: split_name
    for split_name, run_ids in exact_run_ids.items()
    for run_id in run_ids
}
all_assigned_ids = [run_id for run_ids in exact_run_ids.values() for run_id in run_ids]
assert len(all_assigned_ids) == len(set(all_assigned_ids)) == 10
assert set(all_assigned_ids) == set(run_candidates["run_id"])

for split_name, run_ids in exact_run_ids.items():
    print(f"{split_name}:")
    for run_id in run_ids:
        print(f"  {run_id}")

train:
  89cdc34b-e02b-43b4-9284-144276df508a
  d88b15dd-1915-43ca-90ef-69f31ff4d9c1
  ec61755d-b5be-42ac-875d-3123e92add7a
  2e9f4457-2a7f-4647-87ed-b86ac6343d33
  f24e9c1a-f11e-405c-b663-45fa6e25c405
validation:
  fac82c2e-545a-402a-80f8-3d1fccb72c68
  373070d1-ba59-4244-88ab-2d44a21f4983
test:
  6835f125-a038-4092-beff-5107ae998b39
  19127a70-e60c-4b47-b3e6-71e89175c174
  70577f8e-1430-4489-8602-2096521ab84e


### 4. Assign every row and create chronologically sorted split copies

**What the cell does:** Maps each complete run to one split, verifies full assignment, and creates train/validation/test copies sorted by machine, timestamp, run, and segment.  
**Why it is important:** Whole-run assignment prevents leakage while deterministic sorting preserves temporal order inside each machine.  
**What to understand:** Row totals across the three copies equal the input exactly; no feature or target value is transformed.

In [4]:
split_work["_split"] = split_work["run_id"].map(run_to_split)
assert split_work["_split"].notna().all(), "Some rows were not assigned to a split."
assert set(split_work["_split"]) == {"train", "validation", "test"}

sort_columns = ["machine_id", "_timestamp_dt", "run_id", "segment_id", "_source_row"]
split_analysis = {}
split_outputs = {}
for split_name in ["train", "validation", "test"]:
    split_frame = split_work.loc[split_work["_split"].eq(split_name)].sort_values(sort_columns, kind="mergesort").copy()
    split_analysis[split_name] = split_frame
    split_outputs[split_name] = split_frame[original_columns].copy()

assert sum(len(frame) for frame in split_analysis.values()) == len(split_work)
assert set().union(*(set(frame["_source_row"]) for frame in split_analysis.values())) == set(range(len(split_work)))

for split_name, frame in split_outputs.items():
    print(f"{split_name}: {len(frame):,} rows, {frame['run_id'].nunique()} runs, {frame['machine_id'].nunique()} machines")

train: 14,993 rows, 5 runs, 3 machines
validation: 1,872 rows, 2 runs, 2 machines
test: 9,171 rows, 3 runs, 3 machines


### 5. Assert run separation, row separation, classes, and chronology

**What the cell does:** Performs strong set and temporal assertions, including latest-run test membership and per-machine train-before-test checks.  
**Why it is important:** These checks directly guard against run leakage, duplicate rows, incomplete assignment, temporal inversion, and unusable single-class splits.  
**What to understand:** Passing assertions prove structural split safety; they do not guarantee that class rates or machine coverage are ideal.

In [5]:
train_runs = set(exact_run_ids["train"])
validation_runs = set(exact_run_ids["validation"])
test_runs = set(exact_run_ids["test"])
assert train_runs.isdisjoint(validation_runs)
assert train_runs.isdisjoint(test_runs)
assert validation_runs.isdisjoint(test_runs)

row_sets = {
    split_name: set(map(tuple, frame[row_identity_columns].itertuples(index=False, name=None)))
    for split_name, frame in split_outputs.items()
}
assert row_sets["train"].isdisjoint(row_sets["validation"])
assert row_sets["train"].isdisjoint(row_sets["test"])
assert row_sets["validation"].isdisjoint(row_sets["test"])
assert len(row_sets["train"] | row_sets["validation"] | row_sets["test"]) == len(split_work)

for split_name, frame in split_analysis.items():
    assert frame["slowdown_in_5min"].eq(1).any() and frame["slowdown_in_5min"].eq(0).any()
    for _, machine_rows in frame.groupby("machine_id", sort=False):
        assert machine_rows["_timestamp_dt"].is_monotonic_increasing
assert split_analysis["test"]["machine_id"].nunique() == 3

latest_selected_runs = (
    run_candidates.sort_values(["machine_id", "_start_dt", "run_id"])
    .groupby("machine_id", as_index=False).tail(1)
)
assert set(latest_selected_runs["run_id"]) <= test_runs

stage_order = {"train": 0, "validation": 1, "test": 2}
run_candidates["_split"] = run_candidates["run_id"].map(run_to_split)
for machine_id, machine_runs in run_candidates.sort_values(["machine_id", "_start_dt"]).groupby("machine_id"):
    stages = machine_runs["_split"].map(stage_order).tolist()
    assert stages == sorted(stages), f"Chronology violated for machine {machine_id}"
    train_machine = split_analysis["train"].loc[split_analysis["train"]["machine_id"].eq(machine_id)]
    test_machine = split_analysis["test"].loc[split_analysis["test"]["machine_id"].eq(machine_id)]
    if len(train_machine) and len(test_machine):
        assert train_machine["_timestamp_dt"].max() < test_machine["_timestamp_dt"].min()

run_separation_valid = True
chronology_valid = True
print("All run, row, class, machine-coverage, sorting, and chronology assertions passed.")

All run, row, class, machine-coverage, sorting, and chronology assertions passed.


### 6. Build overall, machine, and run summary reports

**What the cell does:** Calculates row proportions, class balance, coverage, segment counts, and time ranges for each split and its machines/runs.  
**Why it is important:** Final reports make the split composition and validation limitation visible before modeling.  
**What to understand:** Zero-row machine combinations remain in the machine report so missing validation coverage is explicit rather than hidden.

In [6]:
total_input_rows = len(split_work)
summary_records = []
run_records = []
for split_name, frame in split_analysis.items():
    positive_rows = int(frame["slowdown_in_5min"].eq(1).sum())
    negative_rows = int(frame["slowdown_in_5min"].eq(0).sum())
    summary_records.append({
        "split": split_name,
        "total_rows": int(len(frame)),
        "percentage_of_total_rows": round(len(frame) / total_input_rows * 100, 4),
        "positive_rows": positive_rows,
        "negative_rows": negative_rows,
        "positive_rate": round(positive_rows / len(frame) * 100, 4),
        "number_of_machines": int(frame["machine_id"].nunique()),
        "number_of_runs": int(frame["run_id"].nunique()),
        "number_of_segments": int(frame["segment_id"].nunique()),
        "minimum_timestamp": frame["_timestamp_dt"].min().isoformat(),
        "maximum_timestamp": frame["_timestamp_dt"].max().isoformat(),
    })
    for (machine_id, run_id), run_data in frame.groupby(["machine_id", "run_id"]):
        run_positive = int(run_data["slowdown_in_5min"].eq(1).sum())
        run_negative = int(run_data["slowdown_in_5min"].eq(0).sum())
        run_records.append({
            "split": split_name,
            "machine_id": machine_id,
            "run_id": run_id,
            "start_timestamp": run_data["_timestamp_dt"].min().isoformat(),
            "end_timestamp": run_data["_timestamp_dt"].max().isoformat(),
            "total_rows": int(len(run_data)),
            "positive_rows": run_positive,
            "negative_rows": run_negative,
            "positive_rate": round(run_positive / len(run_data) * 100, 4),
        })

final_split_summary = pd.DataFrame(summary_records)
final_split_by_run = pd.DataFrame(run_records).sort_values(["split", "machine_id", "start_timestamp"])

machine_records = []
all_machines = sorted(split_work["machine_id"].unique())
for split_name, frame in split_analysis.items():
    for machine_id in all_machines:
        machine_data = frame.loc[frame["machine_id"].eq(machine_id)]
        positives = int(machine_data["slowdown_in_5min"].eq(1).sum())
        negatives = int(machine_data["slowdown_in_5min"].eq(0).sum())
        machine_records.append({
            "split": split_name,
            "machine_id": machine_id,
            "total_rows": int(len(machine_data)),
            "positive_rows": positives,
            "negative_rows": negatives,
            "positive_rate": round(positives / len(machine_data) * 100, 4) if len(machine_data) else np.nan,
            "number_of_runs": int(machine_data["run_id"].nunique()),
        })
final_split_by_machine = pd.DataFrame(machine_records)

display(final_split_summary)
print("By machine:")
display(final_split_by_machine)
print("By run:")
display(final_split_by_run)

,split,total_rows,percentage_of_total_rows,positive_rows,negative_rows,positive_rate,number_of_machines,number_of_runs,number_of_segments,minimum_timestamp,maximum_timestamp
0,train,14993,57.5857,1082,13911,7.2167,3,5,21,2026-07-23T15:00:46.964000+00:00,2026-07-28T17:44:18.127000+00:00
1,validation,1872,7.1900,710,1162,37.9274,2,2,8,2026-07-27T22:57:32.661000+00:00,2026-07-29T09:35:10.620000+00:00
2,test,9171,35.2243,1851,7320,20.1832,3,3,4,2026-07-25T14:45:59.803000+00:00,2026-07-29T13:40:23.654000+00:00


By machine:


,split,machine_id,total_rows,positive_rows,negative_rows,positive_rate,number_of_runs
0,train,0890dcc046c079acc4de4202,8484,256,8228,3.0174,1
1,train,7232bc533c21ce408d45d473,2467,373,2094,15.1196,2
2,train,a0f8c86097e55fbfa506d057,4042,453,3589,11.2073,2
3,validation,0890dcc046c079acc4de4202,0,0,0,NaN,0
4,validation,7232bc533c21ce408d45d473,1611,710,901,44.0720,1
5,validation,a0f8c86097e55fbfa506d057,261,0,261,0.0000,1
6,test,0890dcc046c079acc4de4202,3449,344,3105,9.9739,1
7,test,7232bc533c21ce408d45d473,119,105,14,88.2353,1
8,test,a0f8c86097e55fbfa506d057,5603,1402,4201,25.0223,1


By run:


,split,machine_id,run_id,start_timestamp,end_timestamp,total_rows,positive_rows,negative_rows,positive_rate
7,test,0890dcc046c079acc4de4202,6835f125-a038-4092-beff-5107ae998b39,2026-07-25T14:45:59.803000+00:00,2026-07-25T16:40:54.978000+00:00,3449,344,3105,9.9739
8,test,7232bc533c21ce408d45d473,19127a70-e60c-4b47-b3e6-71e89175c174,2026-07-29T13:35:20.985000+00:00,2026-07-29T13:40:23.654000+00:00,119,105,14,88.2353
9,test,a0f8c86097e55fbfa506d057,70577f8e-1430-4489-8602-2096521ab84e,2026-07-28T17:22:42.771000+00:00,2026-07-28T21:12:01.059000+00:00,5603,1402,4201,25.0223
0,train,0890dcc046c079acc4de4202,89cdc34b-e02b-43b4-9284-144276df508a,2026-07-23T15:00:46.964000+00:00,2026-07-23T19:58:08.855000+00:00,8484,256,8228,3.0174
1,train,7232bc533c21ce408d45d473,d88b15dd-1915-43ca-90ef-69f31ff4d9c1,2026-07-28T10:40:53.862000+00:00,2026-07-28T13:05:14.375000+00:00,1165,311,854,26.6953
2,train,7232bc533c21ce408d45d473,ec61755d-b5be-42ac-875d-3123e92add7a,2026-07-28T15:15:18.443000+00:00,2026-07-28T17:44:18.127000+00:00,1302,62,1240,4.7619
3,train,a0f8c86097e55fbfa506d057,2e9f4457-2a7f-4647-87ed-b86ac6343d33,2026-07-27T12:17:19.031000+00:00,2026-07-27T12:29:16.187000+00:00,182,0,182,0.0000
4,train,a0f8c86097e55fbfa506d057,f24e9c1a-f11e-405c-b663-45fa6e25c405,2026-07-27T13:16:45.456000+00:00,2026-07-27T15:27:24.379000+00:00,3860,453,3407,11.7358
5,validation,7232bc533c21ce408d45d473,fac82c2e-545a-402a-80f8-3d1fccb72c68,2026-07-29T07:39:50.103000+00:00,2026-07-29T09:35:10.620000+00:00,1611,710,901,44.0720
6,validation,a0f8c86097e55fbfa506d057,373070d1-ba59-4244-88ab-2d44a21f4983,2026-07-27T22:57:32.661000+00:00,2026-07-27T23:06:14.948000+00:00,261,0,261,0.0000


### 7. Save split datasets and reports, then validate them

**What the cell does:** Writes the three split CSVs and three reports, reads them back, and repeats critical row, schema, class, and run-disjointness checks.  
**Why it is important:** On-disk validation ensures serialization did not lose rows, alter columns, or mix runs.  
**What to understand:** The saved datasets contain the original feature schema exactly, with no normalization, imputation, balancing, selection, or modeling.

In [7]:
TRAIN_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
SUMMARY_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

split_outputs["train"].to_csv(TRAIN_OUTPUT_PATH, index=False)
split_outputs["validation"].to_csv(VALIDATION_OUTPUT_PATH, index=False)
split_outputs["test"].to_csv(TEST_OUTPUT_PATH, index=False)
final_split_summary.to_csv(SUMMARY_OUTPUT_PATH, index=False)
final_split_by_machine.to_csv(MACHINE_OUTPUT_PATH, index=False)
final_split_by_run.to_csv(RUN_OUTPUT_PATH, index=False)

saved_splits = {
    "train": pd.read_csv(TRAIN_OUTPUT_PATH),
    "validation": pd.read_csv(VALIDATION_OUTPUT_PATH),
    "test": pd.read_csv(TEST_OUTPUT_PATH),
}
saved_summary = pd.read_csv(SUMMARY_OUTPUT_PATH)
saved_machine = pd.read_csv(MACHINE_OUTPUT_PATH)
saved_run = pd.read_csv(RUN_OUTPUT_PATH)

assert sum(len(frame) for frame in saved_splits.values()) == len(feature_input)
assert all(frame.columns.tolist() == original_columns for frame in saved_splits.values())
assert set(saved_splits["train"]["run_id"]).isdisjoint(saved_splits["validation"]["run_id"])
assert set(saved_splits["train"]["run_id"]).isdisjoint(saved_splits["test"]["run_id"])
assert set(saved_splits["validation"]["run_id"]).isdisjoint(saved_splits["test"]["run_id"])
assert all(frame["slowdown_in_5min"].nunique() == 2 for frame in saved_splits.values())
assert saved_splits["test"]["machine_id"].nunique() == 3
assert len(saved_summary) == 3 and len(saved_machine) == 9 and len(saved_run) == 10

print(f"Created {TRAIN_OUTPUT_PATH} ({len(saved_splits['train']):,} rows)")
print(f"Created {VALIDATION_OUTPUT_PATH} ({len(saved_splits['validation']):,} rows)")
print(f"Created {TEST_OUTPUT_PATH} ({len(saved_splits['test']):,} rows)")
print(f"Created {SUMMARY_OUTPUT_PATH}")
print(f"Created {MACHINE_OUTPUT_PATH}")
print(f"Created {RUN_OUTPUT_PATH}")

Created /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/data/modeling/train.csv (14,993 rows)
Created /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/data/modeling/validation.csv (1,872 rows)
Created /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/data/modeling/test.csv (9,171 rows)
Created /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/reports/final_split_summary.csv
Created /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/reports/final_split_by_machine.csv
Created /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/reports/final_split_by_run.csv


### 8. Verify input integrity and print the final handoff

**What the cell does:** Recomputes both input checksums and prints split sizes, rates, machine/run coverage, structural validity, and the main validation limitation.  
**Why it is important:** The final handoff must prove non-modification and clearly communicate limitations before modeling begins.  
**What to understand:** A valid split is structurally safe, but the small, high-positive validation set remains a source of uncertainty.

In [8]:
input_hashes_after = {path: sha256_file(path) for path in input_paths}
inputs_unchanged = input_hashes_before == input_hashes_after
assert inputs_unchanged, "An input file changed during final splitting."

print("FINAL SPLIT REPORT")
for row in final_split_summary.itertuples(index=False):
    print(
        f"{row.split}: {row.total_rows:,} rows ({row.percentage_of_total_rows:.2f}%), "
        f"positive rate={row.positive_rate:.2f}%, machines={row.number_of_machines}, "
        f"runs={row.number_of_runs}, segments={row.number_of_segments}"
    )
print(f"Run separation valid: {run_separation_valid}")
print(f"Chronology valid within machines: {chronology_valid}")
print("Test contains all three machines and the latest selected run for every machine.")
print("Main limitation: validation has only 1,872 rows, covers two machines, and has a 37.93% positive rate.")
print(f"Input feature dataset and run report remained unchanged: {inputs_unchanged}")
print("No normalization, imputation, balancing, feature selection, or model training was performed.")

FINAL SPLIT REPORT
train: 14,993 rows (57.59%), positive rate=7.22%, machines=3, runs=5, segments=21
validation: 1,872 rows (7.19%), positive rate=37.93%, machines=2, runs=2, segments=8
test: 9,171 rows (35.22%), positive rate=20.18%, machines=3, runs=3, segments=4
Run separation valid: True
Chronology valid within machines: True
Test contains all three machines and the latest selected run for every machine.
Main limitation: validation has only 1,872 rows, covers two machines, and has a 37.93% positive rate.
Input feature dataset and run report remained unchanged: True
No normalization, imputation, balancing, feature selection, or model training was performed.
